In [13]:
import numpy as np
import torch
from torch import nn
import pandas as pd
import os
import sys
import copy
import matplotlib.pyplot as plt

PATH = ""
# PATH = "storage"

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [14]:
from PIL import Image
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms
from sklearn.model_selection import train_test_split
from reetoolbox.dataloaders import *

import pandas as pd
import numpy as np
import copy
import time
from reetoolbox.metrics import adversarial_pr_auc, adversarial_roc_auc, adversarial_accuracy
from reetoolbox.optimisers import PGD, StochasticSearch
from reetoolbox.image_evaluator import Evaluator

import matplotlib.pyplot as plt
import seaborn as sns

import os
from huggingface_hub import login, hf_hub_download
import timm
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models
from timm.layers import SwiGLUPacked

In [15]:
github_path = os.path.join(PATH, "Models", "GitHub", "EXAONEPath")
sys.path.append(github_path)
from vision_transformer import VisionTransformer
from huggingface_hub import login, hf_hub_download
import os
import sys
import timm
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models
from timm.layers import SwiGLUPacked
from transformers import AutoModel, AutoImageProcessor
from transformers import AutoImageProcessor, AutoModelForImageClassification, ResNetForImageClassification

In [16]:
from reetoolbox.loadmodels import *
from reetoolbox.eval_funcs import *
# — login once per session —
hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    raise RuntimeError("Please set HF_TOKEN in your environment")
login(token=hf_token, add_to_git_credential=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [17]:
from reetoolbox.transforms import (
    PixelTransform,          # → eval_pixel_optimiser_params, eval_pixel_transform_params
    StainTransform,          # → eval_stain_optimiser_params, eval_stain_transform_params
    MeanTransform,           # → eval_mean_optimiser_params, eval_mean_transform_params
    RotateTransform,         # → eval_rotate_optimiser_params, eval_rotate_transform_params
    CropTransform,           # → eval_crop_optimiser_params, eval_crop_transform_params
    BlurTransform,           # → eval_blur_optimiser_params, eval_blur_transform_params
    ZoomInTransform,         # → eval_zoom_in_optimiser_params, eval_zoom_in_transform_params
    ZoomOutTransform,        # → eval_zoom_out_optimiser_params, eval_zoom_out_transform_params
    HEDTransform,            # → eval_hed_optimiser_params, eval_hed_transform_params
    RandomStainTransform,    # → eval_random_stain_optimiser_params, eval_random_stain_transform_params
    JPEGTransform,           # → eval_jpeg_optimiser_params, eval_jpeg_transform_params
)

from reetoolbox.constants import (
    eval_pixel_optimiser_params,        eval_pixel_transform_params,
    eval_stain_optimiser_params,        eval_stain_transform_params,
    eval_mean_optimiser_params,         eval_mean_transform_params,
    eval_crop_optimiser_params,         eval_crop_transform_params,
    eval_rotate_optimiser_params,       eval_rotate_transform_params,
    eval_blur_optimiser_params,         eval_blur_transform_params,
    eval_zoom_in_optimiser_params,      eval_zoom_in_transform_params,
    eval_zoom_out_optimiser_params,     eval_zoom_out_transform_params,
    eval_hed_optimiser_params,          eval_hed_transform_params,
    eval_random_stain_optimiser_params, eval_random_stain_transform_params,
    eval_jpeg_optimiser_params,         eval_jpeg_transform_params,
)

perturbations = {
    "pixel": {
        "TransformOptimiser": PGD,
        "Transform": PixelTransform,
        "optimiser_params": eval_pixel_optimiser_params,
        "transform_params": eval_pixel_transform_params,
    },
    "mean": {
        "TransformOptimiser": PGD,
        "Transform": MeanTransform,
        "optimiser_params": eval_mean_optimiser_params,
        "transform_params": eval_mean_transform_params,
    },
    "random_stain": {
        "TransformOptimiser": StochasticSearch,
        "Transform": RandomStainTransform,
        "optimiser_params": eval_random_stain_optimiser_params,
        "transform_params": eval_random_stain_transform_params,
    },
    "jpeg": {
        "TransformOptimiser": StochasticSearch,
        "Transform": JPEGTransform,
        "optimiser_params": eval_jpeg_optimiser_params,
        "transform_params": eval_jpeg_transform_params,
    },
    "blur": {
        "TransformOptimiser": StochasticSearch,
        "Transform": BlurTransform,
        "optimiser_params": eval_blur_optimiser_params,
        "transform_params": eval_blur_transform_params,
    },
    "rotate": {
        "TransformOptimiser": StochasticSearch,
        "Transform": RotateTransform,
        "optimiser_params": eval_rotate_optimiser_params,
        "transform_params": eval_rotate_transform_params,
    },
    "zoom_in": {
        "TransformOptimiser": StochasticSearch,
        "Transform": ZoomInTransform,
        "optimiser_params": eval_zoom_in_optimiser_params,
        "transform_params": eval_zoom_in_transform_params,
    },
    "zoom_out": {
        "TransformOptimiser": StochasticSearch,
        "Transform": ZoomOutTransform,
        "optimiser_params": eval_zoom_out_optimiser_params,
        "transform_params": eval_zoom_out_transform_params,
    },
    "crop": {
        "TransformOptimiser": StochasticSearch,
        "Transform": CropTransform,
        "optimiser_params": eval_crop_optimiser_params,
        "transform_params": eval_crop_transform_params,
    },
}

sweep_params = {
    "pixel":        "C",
    "mean":         "C",
    "random_stain": "weights",
    "jpeg":         "quality",
    "blur":         "sigma",          # sweep sigma; set kernel_size = 6*sigma+1 in transform
    "rotate":       "angle",
    "zoom_in":      "scale",
    "zoom_out":     "scale",
    "crop":         "height",         # sweep the side of the crop box (square central crop)
    "hed":          "alpha",          # sweep intensity scaling of HED channels
    "stain":        "C",              # standard L2 constraint for StainTransform
}

# Number of steps for each sweep
n_pixel_steps      = 10
n_mean_steps       = 10
n_randomstain_steps= 10
n_jpeg_steps       = 5
n_blur_steps       = 10
n_rotate_steps     = 20
n_zoomin_steps     = 10
n_zoomout_steps    = 10
n_crop_steps       = 10
n_hed_steps        = 10
n_stain_steps      = 10

# Example: For 224x224 images, crop from 224 (no crop) to 100 (strong crop)
sweep_ranges = {
    "pixel":        (0.00, 0.05),
    "mean":         (0.00, 7.5),
    "random_stain": (0.00, 0.5),
    "jpeg":         (90, 100),
    "blur":         (1, 7),
    "rotate":       (0, 360),
    "zoom_in":      (1.0, 3.0),
    "zoom_out":     (1.0, 0.05),
    "crop":         (224, 50),
    "hed":          (0.8, 1.2),
    "stain":        (0.0, 0.5),      # C: L2 constraint on stain perturbation (adjust as needed)
}

step_sizes = {
    "pixel":        (sweep_ranges["pixel"][1] - sweep_ranges["pixel"][0]) / (n_pixel_steps-1),
    "mean":         (sweep_ranges["mean"][1] - sweep_ranges["mean"][0]) / (n_mean_steps-1),
    "random_stain": (sweep_ranges["random_stain"][1] - sweep_ranges["random_stain"][0]) / (n_randomstain_steps-1),
    "jpeg":         (sweep_ranges["jpeg"][1] - sweep_ranges["jpeg"][0]) / (n_jpeg_steps-1),
    "blur":         (sweep_ranges["blur"][1] - sweep_ranges["blur"][0]) / (n_blur_steps-1),
    "rotate":       (sweep_ranges["rotate"][1] - sweep_ranges["rotate"][0]) / (n_rotate_steps-1),
    "zoom_in":      (sweep_ranges["zoom_in"][1] - sweep_ranges["zoom_in"][0]) / (n_zoomin_steps-1),
    "zoom_out":     -(sweep_ranges["zoom_out"][0] - sweep_ranges["zoom_out"][1]) / (n_zoomout_steps-1),
    "crop":         -(sweep_ranges["crop"][0] - sweep_ranges["crop"][1]) / (n_crop_steps-1),   # decreasing from 224 to 100
    "hed":          (sweep_ranges["hed"][1] - sweep_ranges["hed"][0]) / (n_hed_steps-1),
    "stain":        (sweep_ranges["stain"][1] - sweep_ranges["stain"][0]) / (n_stain_steps-1),
}

In [18]:
model_evals = [
    # # --- Group 1 ---
    # {
    #     "model_name": "GigaPath",
    #     "load_func": lambda: timm.create_model("hf_hub:prov-gigapath/prov-gigapath", pretrained=True).to(device).eval(),
    #     "weight_path": None,
    #     "test_loader_key": "GigaPath",
    #     "output_subdir": "GigaPath",
    # },

    # # --- Group 2 ---
    # {
    #     "model_name": "H-Optimus-0",
    #     "load_func": lambda: timm.create_model("hf-hub:bioptimus/H-optimus-0", pretrained=True, init_values=1e-5, dynamic_img_size=False).to(device).eval(),
    #     "weight_path": None,
    #     "test_loader_key": "H-Optimus-0",
    #     "output_subdir": "H-Optimus-0",
    # },

    # # --- Group 4 ---
    # {
    #     "model_name": "ResNet18",
    #     "load_func": lambda: AutoModelForImageClassification.from_pretrained("microsoft/resnet-18").to(device).eval(),
    #     "weight_path": None,
    #     "test_loader_key": "ResNet18",
    #     "output_subdir": "ResNet18",
    # },
    # {
    #     "model_name": "ResNet50",
    #     "load_func": lambda: ResNetForImageClassification.from_pretrained("microsoft/resnet-50").to(device).eval(),
    #     "weight_path": None,
    #     "test_loader_key": "ResNet50",
    #     "output_subdir": "ResNet50",
    # },

    # {
    #     "model_name": "EXAONEPath",
    #     "load_func": lambda: VisionTransformer.from_pretrained("LGAI-EXAONE/EXAONEPath").to(device).eval(),
    #     "weight_path": None,
    #     "test_loader_key": "EXAONEPath",
    #     "output_subdir": "EXAONEPath",
    # },
    {
        "model_name": "UNI",
        "load_func": lambda: timm.create_model("hf-hub:MahmoodLab/uni", pretrained=True, init_values=1e-5, dynamic_img_size=True).to(device).eval(),
        "weight_path": None,
        "test_loader_key": "UNI",
        "output_subdir": "UNI",
    },
    {
        "model_name": "UNI2",
        "load_func": lambda: timm.create_model("hf-hub:MahmoodLab/UNI2-h", pretrained=True, img_size=224, patch_size=14, depth=24, num_heads=24, init_values=1e-5, embed_dim=1536, mlp_ratio=2.66667*2, no_embed_class=True, mlp_layer=timm.layers.SwiGLUPacked, act_layer=torch.nn.SiLU, reg_tokens=8, dynamic_img_size=True).to(device).eval(),
        "weight_path": None,
        "test_loader_key": "UNI2",
        "output_subdir": "UNI2",
    },
    {
        "model_name": "Hibou-L",
        "load_func": lambda: AutoModel.from_pretrained("histai/hibou-L", trust_remote_code=True).to(device).eval(),
        "weight_path": None,
        "test_loader_key": "Hibou-L",
        "output_subdir": "Hibou-L",
    },

    # # --- Group 3 ---
    # {
    #     "model_name": "H-Optimus-1",
    #     "load_func": lambda: timm.create_model("hf-hub:bioptimus/H-optimus-1", pretrained=True, init_values=1e-5, dynamic_img_size=False).to(device).eval(),
    #     "weight_path": None,
    #     "test_loader_key": "H-Optimus-1",
    #     "output_subdir": "H-Optimus-1",
    # },
    # {
    #     "model_name": "Phikon v1",
    #     "load_func": lambda: AutoModel.from_pretrained("owkin/phikon").to(device).eval(),
    #     "weight_path": None,
    #     "test_loader_key": "Phikon v1",
    #     "output_subdir": "Phikon v1",
    # },
    # {
    #     "model_name": "Phikon v2",
    #     "load_func": lambda: AutoModel.from_pretrained("owkin/phikon-v2").to(device).eval(),
    #     "weight_path": None,
    #     "test_loader_key": "Phikon v2",
    #     "output_subdir": "Phikon v2",
    # },
    # {
    #     "model_name": "Hibou-B",
    #     "load_func": lambda: AutoModel.from_pretrained("histai/hibou-b", trust_remote_code=True).to(device).eval(),
    #     "weight_path": None,
    #     "test_loader_key": "Hibou-B",
    #     "output_subdir": "Hibou-B",
    # },
    # --- Disabled Models ---
    # {
    #     "model_name": "H0-mini",
    #     "load_func": lambda: timm.create_model("hf-hub:bioptimus/H0-mini", pretrained=True, mlp_layer=timm.layers.SwiGLUPacked, act_layer=torch.nn.SiLU).to(device).eval(),
    #     "weight_path": None,
    #     "test_loader_key": "H0-mini",
    #     "output_subdir": "H0-mini",
    # },
    # {
    #     "model_name": "Virchow",
    #     "load_func": lambda: timm.create_model("hf-hub:paige-ai/Virchow", pretrained=True, mlp_layer=timm.layers.SwiGLUPacked, act_layer=torch.nn.SiLU).to(device).eval(),
    #     "weight_path": None,
    #     "test_loader_key": "Virchow",
    #     "output_subdir": "Virchow",
    # },
    # {
    #     "model_name": "Virchow2",
    #     "load_func": lambda: timm.create_model("hf-hub:paige-ai/Virchow2", pretrained=True, mlp_layer=timm.layers.SwiGLUPacked, act_layer=torch.nn.SiLU).to(device).eval(),
    #     "weight_path": None,
    #     "test_loader_key": "Virchow2",
    #     "output_subdir": "Virchow2",
    # },
]

    # {
    #     "model_name": "ResNet34",
    #     "load_func": lambda: AutoModelForImageClassification.from_pretrained("microsoft/resnet-34").to(device).eval(),
    #     "weight_path": None,
    #     "test_loader_key": "ResNet34",
    #     "output_subdir": "ResNet34",
    # },
    # {
    #     "model_name": "ResNet101",
    #     "load_func": lambda: ResNetForImageClassification.from_pretrained("microsoft/resnet-101").to(device).eval(),
    #     "weight_path": None,
    #     "test_loader_key": "ResNet101",
    #     "output_subdir": "ResNet101",
    # },
    # {
    #     "model_name": "ResNet152",
    #     "load_func": lambda: ResNetForImageClassification.from_pretrained("microsoft/resnet-152").to(device).eval(),
    #     "weight_path": None,
    #     "test_loader_key": "ResNet152",
    #     "output_subdir": "ResNet152",
    # },

In [19]:
# Cell 1: Imports & HF-compatible untargeted loss
import os
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.manifold import TSNE
import matplotlib.patches as mpatches
from torchvision import transforms
from reetoolbox.optimisers import untargeted_loss

# def hf_untargeted_loss(outputs, labels):
#     logits = outputs.logits if hasattr(outputs, "logits") else outputs
#     return untargeted_loss(logits, labels)

# --------------------------------------------------------------------------------------------------
# Cell: a universal HF/TIMM‐aware untargeted loss
# --------------------------------------------------------------------------------------------------
import torch
from transformers.modeling_outputs import BaseModelOutputWithPooling
from reetoolbox.optimisers import untargeted_loss

def hf_untargeted_loss(outputs, labels):
    """
    Unwrap any HF / timm output into a plain (B, C) tensor of logits
    so that untargeted_loss(logits, labels) always works.
    """
    # 1) If HF ImageClassifierOutput or similar:
    if hasattr(outputs, "logits"):
        logits = outputs.logits

    # 2) If a BaseModelOutputWithPooling (e.g. timm with num_classes=0 + a pooler):
    elif isinstance(outputs, BaseModelOutputWithPooling) or hasattr(outputs, "pooler_output"):
        logits = outputs.pooler_output

    # 3) If a VisionTransformer‐style return (last_hidden_state [B,seq,dim]):
    elif hasattr(outputs, "last_hidden_state"):
        # take the [CLS] token
        logits = outputs.last_hidden_state[:, 0, :]

    # 4) If they simply returned a tuple whose first element is a tensor:
    elif isinstance(outputs, tuple) and isinstance(outputs[0], torch.Tensor):
        logits = outputs[0]

    # 5) If they already returned a bare tensor:
    elif isinstance(outputs, torch.Tensor):
        logits = outputs

    else:
        raise ValueError(f"Don’t know how to unwrap model output of type {type(outputs)}")

    # 6) If it still has spatial dims (B, C, H, W), global‐avg‐pool to (B, C)
    if logits.ndim == 4:
        logits = torch.nn.functional.adaptive_avg_pool2d(logits, (1,1)).view(logits.size(0), -1)

    # 7) Finally, call your standard gather-based loss:
    return untargeted_loss(logits, labels)


In [20]:
import copy
import torch
import numpy as np
from tqdm import tqdm
from torchvision import transforms
import torch.nn.functional as F

class Evaluator:
    """
    High-level wrapper that attaches a REET attack to your model, and can
    extract feature embeddings (with or without attack) in a uniform way.
    """
    def __init__(
        self,
        model: torch.nn.Module,
        model_name: str,
        TransformOptimiser,
        Transform,
        optimiser_params: dict,
        trans_params: dict,
        criterion,
        device: str = "cuda:0",
        post_transform = None
    ):
        self.model        = model.to(device).eval()
        self.model_name   = model_name
        self.device       = device
        self.criterion    = criterion
        self.trans_params = trans_params
        self.post_transform = post_transform

        # attach the REET attack
        self.attack = TransformOptimiser(
            self.model,
            Transform,
            optimiser_params,
            trans_params,
            criterion=criterion,
            device=device
        )

    def _safe_post_transform(self, imgs):
        """
        Apply post_transform (e.g. normalization / HF processor) if provided.
        """
        if self.post_transform is None:
            return imgs
        if isinstance(imgs, torch.Tensor):
            return imgs
        if isinstance(imgs, (list, tuple)):
            return torch.stack([self._safe_post_transform(i) for i in imgs])
        return self.post_transform(imgs)

    def extract_embeddings(
        self,
        dataloader: torch.utils.data.DataLoader,
        progbar_desc: str = "Extracting embeddings",
        apply_attack: bool = True
    ) -> np.ndarray:
        """
        1) Optionally run the REET attack (on true labels).
        2) Apply post_transform to each image.
        3) Forward through the model, grabbing hidden_states[-1] if available.
        4) Global-average-pool spatial dims to (B, C), then flatten to (N, C).
        Returns: numpy array of shape (N, C).
        """
        self.model.eval()
        all_feats = []
        to_pil = transforms.ToPILImage()

        # save original hyperparameters so we can restore later
        orig_hyp = copy.deepcopy(self.attack.hyperparameters)

        for imgs, labels, *_ in tqdm(dataloader, desc=progbar_desc, leave=False):
            imgs = imgs.to(self.device)
            labels = labels.to(self.device)

            if apply_attack:
                # make sure we can backprop through imgs
                imgs = imgs.clone().detach().requires_grad_(True)
                try:
                    # run REET optimiser; ignore any errors and fall back to clean
                    _, imgs = self.attack.optimise(imgs, targets=labels, reset_weights=True)
                except Exception as e:
                    print(f"[!] {self.model_name} attack failed: {e}. using clean imgs.")
                    imgs = imgs.detach()

            with torch.no_grad():
                processed = []
                for img in imgs.cpu():
                    pil = to_pil(img.clamp(0,1))
                    out = self._safe_post_transform(pil)
                    # HF processors return dicts
                    if isinstance(out, dict) and "pixel_values" in out:
                        out = out["pixel_values"].squeeze(0)
                    processed.append(out)
                batch = torch.stack(processed).to(self.device)

                # forward
                try:
                    out = self.model(batch, output_hidden_states=True)
                except TypeError:
                    out = self.model(batch)

                # grab the final feature map
                if hasattr(out, "hidden_states"):
                    feats = out.hidden_states[-1]
                elif isinstance(out, tuple):
                    feats = out[0]
                else:
                    feats = out

                # global‐avg‐pool spatial dims (B, C, H, W) → (B, C)
                if feats.ndim == 4:
                    feats = F.adaptive_avg_pool2d(feats, (1,1)).view(feats.size(0), -1)
                else:
                    feats = feats.view(feats.size(0), -1)

                all_feats.append(feats.cpu())

        # restore attack hyperparameters
        self.attack.hyperparameters = orig_hyp

        return torch.cat(all_feats, dim=0).numpy()

In [21]:
import os, copy, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.manifold import TSNE

def sweep_embedding_similarity(
    evaluator: Evaluator,
    model_name: str,
    train_emb: np.ndarray,
    test_loader: torch.utils.data.DataLoader,
    perturb_name: str,
    hyperparam_name: str,
    sweep_range: tuple,
    step_size: float,
    output_dir: str,
    tsne_perplexity: int = 30,
    tsne_randstate: int = 42
) -> pd.DataFrame:
    """
    1) Baseline (no attack) on test -> embeddings/similarity
    2) Sweep strengths from sweep_range, record mean sim each
    3) Save stats & similarity plot
    4) Run joint t-SNE on train + *all* test embeddings
    5) Save coords (split only 'train'/'test') + scatter+1σ ellipse plot
    """
    os.makedirs(output_dir, exist_ok=True)
    orig_hyp = copy.deepcopy(evaluator.attack.hyperparameters)

    # Prepare strength list
    step_size = abs(sweep_range[0] - sweep_range[1]) / 10

    if sweep_range[0] > sweep_range[1]:
        strengths = np.arange(sweep_range[0], sweep_range[1] - step_size, -step_size)
    else:
        strengths = np.arange(sweep_range[0], sweep_range[1] + step_size, step_size)
    print(strengths)
    # 1) Baseline test (no attack)
    baseline_emb = evaluator.extract_embeddings(
        test_loader, progbar_desc="Test emb baseline", apply_attack=False
    )
    baseline_sim = cosine_similarity(baseline_emb, train_emb).mean()

    means = [baseline_sim]
    tsne_batches = [
        (0.0, train_emb),
        (0.0, baseline_emb)
    ]

    # 2) Sweep through strengths
    for s in tqdm(strengths, desc=f"Sweeping {perturb_name}"):
        hyp = copy.deepcopy(orig_hyp)
        # ---- custom hyperparam logic ----
        if hyperparam_name == "sigma" and "weight_ranges" in hyp:
            v = float(s)
            hyp["weight_ranges"]["sigma"] = (v, v)
            k = int(math.ceil(6*v+1))
            if k % 2 == 0: k += 1
            hyp["weight_ranges"]["kernel_size"] = (k, k)
            hyp["weight_ranges"].update({
                "corner_x": (0,0), "corner_y": (0,0),
                "height":   (224,224), "width":   (224,224)
            })
        elif hyperparam_name == "angle" and "weight_ranges" in hyp:
            hyp["weight_ranges"]["angle"] = (s, s)
        elif hyperparam_name == "scale" and "weight_ranges" in hyp:
            hyp["weight_ranges"]["scale"] = (s, s)
        elif hyperparam_name == "height" and "weight_ranges" in hyp:
            hyp["weight_ranges"]["height"] = (s, s)
            if "width" in hyp["weight_ranges"]:
                hyp["weight_ranges"]["width"] = (s, s)
        elif hyperparam_name in hyp:
            hyp[hyperparam_name] = s
        elif "weight_ranges" in hyp and hyperparam_name in hyp["weight_ranges"]:
            if hyperparam_name == "quality":
                hyp["weight_ranges"][hyperparam_name] = (s, s)
            else:
                hyp["weight_ranges"][hyperparam_name] = (-s, s)
        else:
            raise KeyError(f"Parameter '{hyperparam_name}' not found.")
        # -----------------------------------

        evaluator.attack.hyperparameters = hyp
        emb = evaluator.extract_embeddings(
            test_loader,
            progbar_desc=f"Test emb {perturb_name}={s:.2f}",
            apply_attack=True
        )
        means.append(cosine_similarity(emb, train_emb).mean())
        tsne_batches.append((float(s), emb))

    # restore original
    evaluator.attack.hyperparameters = orig_hyp

    # 3) Save stats & similarity plot
    all_strengths = np.concatenate(([0.0], strengths))
    df_stats = pd.DataFrame({'strength': all_strengths, 'mean_sim': means})
    stats_path = os.path.join(output_dir, f"{perturb_name}_stats.xlsx")
    df_stats.to_excel(stats_path, index=False)
    print(f"Saved stats -> {stats_path}")

    plt.figure(figsize=(8,4))
    plt.plot(all_strengths, means, marker='o')
    plt.title(f"{model_name}: Cosine Similarity vs {perturb_name}")
    plt.xlabel("strength"); plt.ylabel("mean cosine sim")
    plt.grid(True); plt.tight_layout()
    sim_png = os.path.join(output_dir, f"{perturb_name}_similarity.png")
    # plt.savefig(sim_png); plt.close()
    # print(f"Saved plot  -> {sim_png}")

    # 4) Joint t-SNE
    all_emb = np.vstack([emb for _, emb in tsne_batches])
    tsne_out = TSNE(
        n_components=2,
        perplexity=tsne_perplexity,
        random_state=tsne_randstate
    ).fit_transform(all_emb)

    # 5) Save coords (split only 'train'/'test')
    records, offset = [], 0
    for idx, (strength, emb) in enumerate(tsne_batches):
        split = 'train' if idx == 0 else 'test'
        n = emb.shape[0]
        pts = tsne_out[offset:offset+n]
        for x, y in pts:
            records.append({
                'strength': strength,
                'split':    split,
                'tsne1':    x,
                'tsne2':    y
            })
        offset += n

    tsne_path = os.path.join(output_dir, f"{perturb_name}_tsne.xlsx")
    pd.DataFrame(records).to_excel(tsne_path, index=False)
    print(f"Saved t-SNE coords -> {tsne_path}")

    # 6) Plot scatter + 1σ ellipse + means
    plt.figure(figsize=(10,7))
    ax = plt.gca()
    colors = plt.cm.viridis(np.linspace(0,1,len(tsne_batches)))[::-1]
    offset = 0

    for idx, (strength, emb) in enumerate(tsne_batches):
        label = 'train' if idx==0 else f'test {perturb_name}={strength:.2f}'
        n = emb.shape[0]
        pts = tsne_out[offset:offset+n]

        ax.scatter(pts[:,0], pts[:,1],
                   c=[colors[idx]], label=label,
                   s=10, alpha=0.4)

        if n > 2:
            mu  = pts.mean(axis=0)
            cov = np.cov(pts, rowvar=False)
            vals, vecs = np.linalg.eigh(cov)
            order = vals.argsort()[::-1]
            angle = np.degrees(np.arctan2(*vecs[:,0][::-1]))
            width, height = 2*np.sqrt(vals[order])  # 1σ
            ell = mpatches.Ellipse(mu, width, height,
                                   angle=angle, fill=False,
                                   edgecolor=colors[idx])
            ax.add_patch(ell)
            ax.scatter(mu[0], mu[1], marker='X', s=100,
                       c=[colors[idx]], edgecolor='black',
                       label=f"{label} mean")

        offset += n

    ax.legend(bbox_to_anchor=(1.05,1), loc="upper left")
    ax.set_title(f"t-SNE embeddings under {perturb_name}")
    ax.set_xlabel("Dim 1"); ax.set_ylabel("Dim 2")
    plt.tight_layout()
    tsne_png = os.path.join(output_dir, f"{perturb_name}_tsne.png")
    # plt.savefig(tsne_png); plt.close()
    # print(f"Saved t-SNE plot -> {tsne_png}")

    return df_stats


In [22]:
# Cell 4: batch_embedding_sweep (with criterion restored)
def batch_embedding_sweep(
    dataset_name: str,
    model_eval: dict,
    params: dict,
    perturbations: dict,
    sweep_params: dict,
    sweep_ranges: dict,
    step_sizes: dict,
    model_transforms: dict,
    device: torch.device,
    PATH: str
):
    # 1) load model
    model = model_eval["load_func"]().to(device).eval()
    if model_eval.get("weight_path"):
        model.head.load_state_dict(torch.load(model_eval["weight_path"]))

    # 2) post-transform
    post_tr = model_transforms[ model_eval["test_loader_key"] ]

    # 3) build train loader
    td = create_trainval_dict(
        dataset_class       = params["dataset_class"],
        root_dir            = params["root_dir"],
        batch_size          = params["batch_size"],
        transform           = post_tr,
        trainval_multiplier = params["trainval_multiplier"],
        trainval_size       = params["trainval_size"],
        **params["train_extra"]
    )
    train_loader = td["train"]

    # 4) build test loader & report
    _, test_loader = build_dataset(
        dataset_class   = params["dataset_class"],
        root_dir        = params["test_root"],
        batch_size      = params["batch_size"],
        test_multiplier = params["test_multiplier"],
        transform       = post_tr,
        **params["test_extra"],
        shuffle         = False
    )
    print(f"[{dataset_name}] Test batches: {len(test_loader)}")

    # 5) init evaluator (with criterion)
    first_cfg = next(iter(perturbations.values()))
    evaluator = Evaluator(
        model              = model,
        model_name         = model_eval["model_name"],
        TransformOptimiser = first_cfg["TransformOptimiser"],
        Transform          = first_cfg["Transform"],
        optimiser_params   = first_cfg["optimiser_params"],
        trans_params       = first_cfg["transform_params"],
        criterion          = hf_untargeted_loss,
        device             = device,
        post_transform     = post_tr
    )

    # 6) extract train embeddings WITHOUT attack
    train_emb = evaluator.extract_embeddings(
        train_loader,
        progbar_desc="Train emb",
        apply_attack=False
    )
    print(f"[{model_eval['model_name']}] train_emb shape: {train_emb.shape}")

    # 7) make output dir
    outdir = os.path.join(PATH, "Stats", dataset_name, model_eval["model_name"], "embeddings")
    os.makedirs(outdir, exist_ok=True)

    # 8) sweep each perturbation
    for name, cfg in perturbations.items():
        print(f"→ {model_eval['model_name']} embedding sweep '{name}'")
        # re-assign attack (with criterion)
        evaluator.attack = cfg["TransformOptimiser"](
            model,
            cfg["Transform"],
            cfg["optimiser_params"],
            cfg["transform_params"],
            criterion = hf_untargeted_loss,
            device    = device
        )
        sweep_embedding_similarity(
            evaluator       = evaluator,
            model_name      = model_eval["model_name"],
            train_emb       = train_emb,
            test_loader     = test_loader,
            perturb_name    = name,
            hyperparam_name = sweep_params[name],
            sweep_range     = sweep_ranges[name],
            step_size       = step_sizes[name],
            output_dir      = outdir
        )

# Cell 5: run_all_model_embedding_sweeps
def run_all_model_embedding_sweeps(
    dataset_name: str,
    model_evals: list,
    params: dict,
    perturbations: dict,
    sweep_params: dict,
    sweep_ranges: dict,
    step_sizes: dict,
    model_transforms: dict,
    device: torch.device,
    PATH: str
):
    """
    dataset_name: key into DATASET_PARAMS
    params: DATASET_PARAMS[dataset_name]
    """
    # pull out this dataset's config
    cfg = params

    for me in model_evals:
        print(f"\n=== Embedding sweeps for {me['model_name']} on {dataset_name} ===")
        batch_embedding_sweep(
            dataset_name      = dataset_name,
            model_eval        = me,
            params            = cfg,
            perturbations     = me.get("perturbations", perturbations),
            sweep_params      = sweep_params,
            sweep_ranges      = sweep_ranges,
            step_sizes        = step_sizes,
            model_transforms  = model_transforms,
            device            = device,
            PATH              = PATH
        )

In [23]:
DATASET_PARAMS = {
    "NCT": {
        "dataset_class":      NCTDataSet,
        "root_dir":           os.path.join(PATH,'data','NCT','NCT-CRC-HE-100K'),
        "test_root":          os.path.join(PATH,'data','NCT','CRC-VAL-HE-7K'),
        "batch_size":         16,
        "trainval_multiplier":0.03666,
        "trainval_size":      0.002,
        "test_multiplier":    0.075,
        "train_extra":        dict(classification_mode='tum_vs_all'),
        "test_extra":         dict(classification_mode='tum_vs_all'),
    },
    "PanNuke": {
        "dataset_class":      PanNukeDataset,
        "root_dir":           os.path.join(PATH,'data','PanNuke'),
        "test_root":          os.path.join(PATH,'data','PanNuke'),
        "batch_size":         16,
        "trainval_multiplier":0.465,
        "trainval_size":      0.002,
        "test_multiplier":    0.1,
        "train_extra":        dict(folds=(1,2), min_positive=5),
        "test_extra":         dict(folds=(3,), min_positive=5),
    },
    "PANDA": {
        "dataset_class":      PandaDataset,
        "root_dir":           os.path.join(PATH,'data','PANDA'),
        "test_root":          os.path.join(PATH,'data','PANDA'),
        "batch_size":         16,
        "trainval_multiplier":0.0565,
        "trainval_size":      0.002,
        "test_multiplier":    0.069,
        "train_extra":        dict(split='train'),
        "test_extra":         dict(split='test'),
    },
    "PatchCamelyon": {
        "dataset_class":       PatchCamelyonDataset,
        "root_dir":            os.path.join(PATH, "data", "PatchCamelyon"),
        "test_root":           os.path.join(PATH, "data", "PatchCamelyon"),
        "batch_size":          16,
        # subsample fractions (tune for your runtime):
        "trainval_multiplier": 0.0128,   # proportion of the *train* split to use (via create_trainval_dict special-case)
        "trainval_size":       0.001,  # proportion of the *valid* split to use for val
        "test_multiplier":     0.01558,  # proportion of the *test* split to use (via build_dataset)
        # extras:
        "train_extra":         {},                 # special-case loader already uses split='train' and 'valid'
        "test_extra":          dict(split="test"), # ensure the test HDF5/CSV are used
},

}

dataset_name = "PatchCamelyon"
cfg = DATASET_PARAMS[dataset_name]

In [24]:
# run_all_model_embedding_sweeps(
#     dataset_name      = dataset_name,
#     model_evals       = model_evals,
#     params            = cfg,
#     perturbations     = perturbations,
#     sweep_params      = sweep_params,
#     sweep_ranges      = sweep_ranges,
#     step_sizes        = step_sizes,
#     model_transforms  = model_transforms,
#     device            = device,
#     PATH              = PATH
# )

In [25]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

def visualize_and_save_embedding_sweeps(
    embeddings_dir: str,
    perturbations: list[str] | None = None,
    plot_ellipse: bool = True,
    ellipse_std: float = 1.0,
    normalize_y: bool = False,
    fixed_y_range: bool = False,
    tsne_step: int = 1,                         # new: plot every nth TSNE group
    figsize_similarity: tuple[int,int] = (8,4),
    figsize_tsne: tuple[int,int] = (10,7)
):
    """
    Reads *_stats.xlsx and *_tsne.xlsx from embeddings_dir and for each perturb:
      • plots & saves full mean-similarity curve,
      • plots & saves t-SNE scatter with optional ellipse selecting every tsne_step group.
    """
    model_name = os.path.basename(os.path.dirname(embeddings_dir)).lower()
    stats_paths = sorted(glob.glob(os.path.join(embeddings_dir, "*_stats.xlsx")))
    tsne_map    = {
        os.path.basename(p).split("_tsne.xlsx")[0]: p
        for p in glob.glob(os.path.join(embeddings_dir, "*_tsne.xlsx"))
    }

    drop_zero   = {"crop", "jpeg", "zoom_out", "zoom_in"}
    invert_axis = {"crop", "zoom_out", "jpeg"}
    baseline_x  = {"jpeg": 100.0, "zoom_out": 1.0}

    for stat_path in stats_paths:
        perturb = os.path.basename(stat_path).removesuffix("_stats.xlsx")
        if perturbations and perturb not in perturbations:
            continue

        # load stats
        df    = pd.read_excel(stat_path)
        x_all = pd.to_numeric(df.iloc[:,0], errors="coerce").to_numpy()
        y_all = pd.to_numeric(df.iloc[:,1], errors="coerce").to_numpy()

        # handle baseline for jpeg/zoom_out
        if perturb in baseline_x:
            baseline_y = float(y_all[x_all==0.0][0])
            mask = x_all != 0.0
            x_rest, y_rest = x_all[mask], y_all[mask]
            x = np.concatenate(([baseline_x[perturb]], x_rest))
            y = np.concatenate(([baseline_y], y_rest))
        else:
            mask = x_all!=0.0 if perturb in drop_zero else np.ones_like(x_all, bool)
            x, y = x_all[mask], y_all[mask]
            baseline_y = y[0]

        # normalize if requested
        if normalize_y:
            y = y / baseline_y

        # — similarity plot —
        fig, ax = plt.subplots(figsize=figsize_similarity)
        if perturb in baseline_x:
            ax.scatter([x[0]], [y[0]], marker="o", s=50, label="baseline")
            ax.plot(x[1:], y[1:], marker="o", linewidth=1)
        else:
            ax.plot(x, y, marker="o", linewidth=1)

        ylabel = "normalized cosine sim" if normalize_y else "mean cosine sim"
        ax.set_title(f"{model_name.upper()} — {perturb}: Cosine similarity")
        ax.set_xlabel("strength")
        ax.set_ylabel(ylabel)
        ax.grid(True)

        if fixed_y_range:
            ax.set_ylim(0.0, 1.0)

        if perturb in invert_axis and x.size:
            ax.invert_xaxis()

        sim_fn = f"{perturb}_similarity_{model_name}.png"
        sim_out = os.path.join(embeddings_dir, sim_fn)
        fig.tight_layout()
        fig.savefig(sim_out)
        plt.close(fig)
        print(f"Saved similarity → {sim_fn}")

        # — t-SNE plot —
        tsne_path = tsne_map.get(perturb)
        if not tsne_path:
            print(f"Skipping t-SNE for '{perturb}': coords missing.")
            continue

        df_tsne = pd.read_excel(tsne_path).astype({"tsne1": float, "tsne2": float})
        all_groups = list(df_tsne.groupby(["split","strength"], sort=False))

        # always keep first two (train baseline, test baseline), then every tsne_step thereafter
        selected = all_groups[:2] + [
            grp for idx, grp in enumerate(all_groups[2:], start=2)
            if ((idx-2) % tsne_step) == 0 or idx == len(all_groups)-1
        ]
        n = len(selected)
        cmap = plt.cm.viridis_r

        fig, ax = plt.subplots(figsize=figsize_tsne)
        for idx, ((split, strg), sub) in enumerate(selected):
            pts = sub[["tsne1","tsne2"]].to_numpy()
            col = cmap(idx/(n-1)) if n>1 else cmap(0.0)
            ax.scatter(pts[:,0], pts[:,1],
                       c=[col],
                       label=f"{split} {strg:.2f}",
                       s=10, alpha=0.4)
            if plot_ellipse and pts.shape[0]>2:
                mu  = pts.mean(axis=0)
                cov = np.cov(pts, rowvar=False)
                vals, vecs = np.linalg.eigh(cov)
                order = vals.argsort()[::-1]
                angle = np.degrees(np.arctan2(*vecs[:,0][::-1]))
                width, height = 2*ellipse_std*np.sqrt(vals[order])
                ell = mpatches.Ellipse(mu, width, height,
                                       angle=angle,
                                       fill=False,
                                       edgecolor=col,
                                       lw=2)
                ax.add_patch(ell)
                ax.scatter(mu[0], mu[1],
                           marker="X", s=100,
                           c=[col], edgecolor="black")

        ax.legend(bbox_to_anchor=(1.05,1), loc="upper left")
        ax.set_title(f"{model_name.upper()} — {perturb}: t-SNE embeddings")
        ax.set_xlabel("Dim 1")
        ax.set_ylabel("Dim 2")

        tsne_fn = f"{perturb}_tsne_{model_name}.png"
        tsne_out = os.path.join(embeddings_dir, tsne_fn)
        fig.tight_layout()
        fig.savefig(tsne_out)
        plt.close(fig)
        print(f"Saved t-SNE      → {tsne_fn}")

In [28]:
import os
os.path.join(PATH, "Stats", dataset_name)
# models = os.listdir(r"C:\Users\Dhyey\Documents\University\MSc\CS907 Dissertation Project\Tutorials\Stats\NCT")
models = os.listdir(os.path.join(PATH, "Stats", dataset_name))
del models[0]

for model in models:
    visualize_and_save_embedding_sweeps(
        embeddings_dir=os.path.join(PATH, "stats", dataset_name, model, "embeddings"),
        perturbations=None,       # or e.g. ["brightness","contrast"]
        plot_ellipse=True,
        ellipse_std=1.0,
        normalize_y=True,
        fixed_y_range=False,
        tsne_step=2,
    )

Saved similarity → blur_similarity_exaonepath.png
Saved t-SNE      → blur_tsne_exaonepath.png
Saved similarity → crop_similarity_exaonepath.png
Saved t-SNE      → crop_tsne_exaonepath.png
Saved similarity → jpeg_similarity_exaonepath.png
Saved t-SNE      → jpeg_tsne_exaonepath.png
Saved similarity → mean_similarity_exaonepath.png
Saved t-SNE      → mean_tsne_exaonepath.png
Saved similarity → pixel_similarity_exaonepath.png
Saved t-SNE      → pixel_tsne_exaonepath.png
Saved similarity → random_stain_similarity_exaonepath.png
Saved t-SNE      → random_stain_tsne_exaonepath.png
Saved similarity → rotate_similarity_exaonepath.png
Saved t-SNE      → rotate_tsne_exaonepath.png
Saved similarity → zoom_in_similarity_exaonepath.png
Saved t-SNE      → zoom_in_tsne_exaonepath.png
Saved similarity → zoom_out_similarity_exaonepath.png
Saved t-SNE      → zoom_out_tsne_exaonepath.png
Saved similarity → blur_similarity_gigapath.png
Saved t-SNE      → blur_tsne_gigapath.png
Saved similarity → crop_simi